In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from transformers import pipeline
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

generator = pipeline("mask-generation", model="facebook/sam-vit-huge", device=0, framework="pt")
image_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/real/0a1f851b-a3fe-4609-b781-3fd268926582.png"
image_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/visualization/top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all-uids/real/0d8f56ef-0914-491c-944b-216f4022e929.png"
image_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/visualization/top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all-uids/real/1bcdc1ac-4ae5-481d-b279-086e7f9f0e4f.png"

In [ ]:
# 将 mask 叠加在原图上可视化
def visualize_masks_on_image(image, masks, alpha=0.5, max_masks=10):
    """
    Args:
        image: PIL Image 对象
        masks: mask 列表
        alpha: mask 的透明度 (0-1)
        max_masks: 最多显示的 mask 数量
    """
    img_array = np.array(image)
    overlay = img_array.copy()
    colors = plt.cm.tab20(np.linspace(0, 1, min(len(masks), max_masks)))
    
    for idx, mask in enumerate(masks[:max_masks]):
        if isinstance(mask, Image.Image):
            mask_array = np.array(mask)
        else:
            mask_array = mask
        
        if mask_array.ndim == 3:
            mask_array = mask_array[:, :, 0] if mask_array.shape[2] == 1 else mask_array[:, :, 0]
        
        if mask_array.max() > 1:
            mask_array = mask_array.astype(float) / 255.0
        
        color = (colors[idx][:3] * 255).astype(int)
        colored_mask = np.zeros_like(img_array)
        colored_mask[mask_array > 0.5] = color
        
        mask_bool = mask_array > 0.5
        overlay[mask_bool] = (1 - alpha) * overlay[mask_bool] + alpha * colored_mask[mask_bool]
    
    return Image.fromarray(overlay.astype(np.uint8))

image = Image.open(image_path).convert("RGB")
outputs = generator(image)
masks = outputs["masks"]
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

axes[0].imshow(image)
axes[0].set_title(os.path.basename(image_path), fontsize=14)
axes[0].axis('off')

overlay_image = visualize_masks_on_image(image, masks, alpha=0.5, max_masks=10)
axes[1].imshow(overlay_image)
axes[1].set_title(f'Image with {min(len(masks), 10)} Masks Overlaid', fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Total masks generated: {len(masks)}")
